<a href="https://colab.research.google.com/github/marcuspaiv/DIO_desafio_Conversando-por-Voz-Com-o-ChatGPT-Utilizando-Whisper-OpenAI-e-Python/blob/main/C%C3%B3pia_de_Assistente_de_Voz_Multi_Idiomas_Com_Whisper_e_ChatGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assistente de Voz com IA Local (GPT4All)

**1. CÓDIGO PARA GRAVAR ÁUDIO**

In [ ]:
language = 'pt'

In [ ]:
!pip install SpeechRecognition
!pip install gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import os
import speech_recognition as sr
from gtts import gTTS

# ============================================
# 1. CÓDIGO PARA GRAVAR ÁUDIO
# ============================================

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks, {type: 'audio/webm'})
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
    """Grava áudio do usuário"""
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])

    file_name = 'request_audio.webm'
    with open(file_name, 'wb') as f:
        f.write(audio)

    return f'/content/{file_name}'

**2. INSTALAR PACOTES NECESSÁRIOS**

In [ ]:
# ============================================
# 2. INSTALAR PACOTES NECESSÁRIOS
# ============================================

print('📦 Instalando pacotes...')
!pip install -q gtts SpeechRecognition pydub gpt4all

📦 Instalando pacotes...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.0 MB/s eta 0:00:00


## **3. GRAVAR E TRANSCREVER ÁUDIO**

In [ ]:
# ============================================
# 3. GRAVAR E TRANSCREVER ÁUDIO
# ============================================

print('\n🎤 Grave sua pergunta (vou gravar por 5 segundos)')
print('⚠️  Quando aparecer "Falando..." comece a falar claramente\n')

# Gravar áudio
record_file = record(sec=5)

# Converter para formato compatível
print('🔄 Processando áudio...')
!ffmpeg -i {record_file} -acodec pcm_s16le -ac 1 -ar 16000 temp_audio.wav -y -loglevel quiet

# Transcrever usando Google Speech Recognition (gratuito)
recognizer = sr.Recognizer()
with sr.AudioFile('temp_audio.wav') as source:
    print('🎙️  Analisando sua gravação...')
    audio_data = recognizer.record(source)

    try:
        transcription = recognizer.recognize_google(audio_data, language='pt-BR')
        print(f'\n✅ Você disse: "{transcription}"')
    except sr.UnknownValueError:
        print('\n❌ Não consegui entender o áudio. Usando pergunta padrão.')
        transcription = "Qual é a capital do Brasil?"
    except sr.RequestError as e:
        print(f'\n❌ Erro no serviço: {e}. Usando pergunta padrão.')
        transcription = "Qual é a capital do Brasil?"



🎤 Grave sua pergunta (vou gravar por 5 segundos)
⚠️  Quando aparecer "Falando..." comece a falar claramente



<IPython.core.display.Javascript object>

🔄 Processando áudio...
🎙️  Analisando sua gravação...

✅ Você disse: "O que é Geologia"


# 4. CARREGAR MODELO LOCAL GRATUITO (GPT4All)

In [ ]:
# ============================================
# 4. CARREGAR MODELO LOCAL GRATUITO (GPT4All)
# ============================================

print('\n🤖 Baixando e carregando modelo de IA gratuito...')
print('⏳ Isso pode levar alguns minutos na primeira execução...')

from gpt4all import GPT4All

# Baixar modelo pequeno e gratuito (cerca de 4GB)
# Você pode escolher outros modelos disponíveis em: https://gpt4all.io/index.html
model_name = "orca-mini-3b-gguf2-q4_0.gguf"

try:
    # Inicializar o modelo (faz download automático na primeira vez)
    model = GPT4All(model_name)
    print('✅ Modelo carregado com sucesso!')

    # Gerar resposta
    print('💭 Gerando resposta...')

    # Prompt em português
    prompt = f"""Responda em português de forma clara e objetiva:

Pergunta: {transcription}

Resposta:"""

    response = model.generate(prompt, max_tokens=150)
    chatgpt_response = response.strip()

    print(f'\n💬 Resposta: {chatgpt_response}')

except Exception as e:
    print(f'⚠️  Erro ao carregar modelo: {e}')
    print('\nUsando resposta simulada...')
    chatgpt_response = f"Você perguntou: '{transcription}'. O modelo local não pôde ser carregado, mas você pode tentar novamente mais tarde."


🤖 Baixando e carregando modelo de IA gratuito...
⏳ Isso pode levar alguns minutos na primeira execução...


Downloading: 100%|██████████| 1.98G/1.98G [00:26<00:00, 73.9MiB/s]
Verifying: 100%|██████████| 1.98G/1.98G [00:04<00:00, 452MiB/s]


✅ Modelo carregado com sucesso!
💭 Gerando resposta...

💬 Resposta: Geologia é o estudo do material da terra, como as minérias, rochas e sedimentos. Ele permite compreender a história natural da Terra, desde o início dos tempos até hoje, mediante a análise de dados geológicos e fisico-geológicos.


**5. CONVERTER RESPOSTA PARA ÁUDIO**

In [ ]:
# ============================================
# 5. CONVERTER RESPOSTA PARA ÁUDIO
# ============================================

print('\n🔊 Convertendo resposta para áudio...')
language = 'pt'
gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)
response_audio = "/content/response_audio.mp3"
gtts_object.save(response_audio)

print('▶️  Reproduzindo resposta...')
display(Audio(response_audio, autoplay=True))


🔊 Convertendo resposta para áudio...
▶️  Reproduzindo resposta...


**` 6. LIMPEZA (opcional)`**

In [ ]:
# ============================================
# 6. LIMPEZA (opcional)
# ============================================

# Remover arquivos temporários
os.system('rm -f temp_audio.wav')
print('\n✨ Processo concluído!')


✨ Processo concluído!
